In [1]:
from sklearn.metrics import accuracy_score

first_round = [
    # Group A
    ("Mexico", "South Africa", "MX"),
    ("South Korea", "Czechia", "MX"),
    # Group B
    ("Canada", "Bosnia and Herzegovina", "CA"),
    ("Qatar", "Switzerland", "US"),
    # Group C
    ("Brazil", "Morocco", "US"),
    ("Haiti", "Scotland", "US"),
    # Group D
    ("USA", "Paraguay", "US"),
    ("Australia", "Türkiye", "CA"),
    # Group E
    ("Germany", "Curaçao", "US"),
    ("Côte d'Ivoire", "Ecuador", "US"),
    # Group F
    ("Netherlands", "Japan", "US"),
    ("Sweden", "Tunisia", "MX"),
    # Group G
    ("Belgium", "Egypt", "US"),
    ("IR Iran", "New Zealand", "US"),
    # Group H
    ("Spain", "Cabo Verde", "US"),
    ("Saudi Arabia", "Uruguay", "US"),
    # Group I
    ("France", "Senegal", "US"),
    ("Iraq", "Norway", "US"),
    # Group J
    ("Argentina", "Algeria", "US"),
    ("Austria", "Jordan", "US"),
    # Group K
    ("Portugal", "Congo DR", "US"),
    ("Uzbekistan", "Colombia", "MX"),
    # Group L
    ("Ghana", "Panama", "CA"),
    ("England", "Croatia", "US"),
]

second_round = [
    # Group A
    ("Czechia", "South Africa", "US"),
    ("Mexico", "South Korea", "MX"),
    # Group B
    ("Switzerland", "Bosnia and Herzegovina", "US"),
    ("Canada", "Qatar", "CA"),
    # Group C
    ("Brazil", "Haiti", "US"),
    ("Scotland", "Morocco", "US"),
    # Group D
    ("Türkiye", "Paraguay", "US"),
    ("USA", "Australia", "US"),
    # Group E
    ("Germany", "Côte d'Ivoire", "CA"),
    ("Ecuador", "Curaçao", "US"),
    # Group F
    ("Netherlands", "Sweden", "US"),
    ("Tunisia", "Japan", "MX"),
    # Group G
    ("Belgium", "IR Iran", "US"),
    ("New Zealand", "Egypt", "CA"),
    # Group H
    ("Spain", "Saudi Arabia", "US"),
    ("Uruguay", "Cabo Verde", "US"),
    # Group I
    ("France", "Iraq", "US"),
    ("Norway", "Senegal", "US"),
    # Group J
    ("Argentina", "Austria", "US"),
    ("Jordan", "Algeria", "US"),
    # Group K
    ("Portugal", "Uzbekistan", "US"),
    ("Colombia", "Congo DR", "MX"),
    # Group L
    ("England", "Ghana", "US"),
    ("Panama", "Croatia", "CA"),
]

third_round = [
    # Group A
    ("Czechia", "Mexico", "MX"),
    ("South Africa", "South Korea", "MX"),
    # Group B
    ("Switzerland", "Canada", "CA"),
    ("Bosnia and Herzegovina", "Qatar", "US"),
    # Group C
    ("Scotland", "Brazil", "US"),
    ("Morocco", "Haiti", "US"),
    # Group D
    ("Türkiye", "USA", "US"),
    ("Paraguay", "Australia", "US"),
    # Group E
    ("Curaçao", "Côte d'Ivoire", "US"),
    ("Ecuador", "Germany", "US"),
    # Group F
    ("Japan", "Sweden", "US"),
    ("Tunisia", "Netherlands", "US"),
    # Group G
    ("Egypt", "IR Iran", "US"),
    ("New Zealand", "Belgium", "CA"),
    # Group H
    ("Uruguay", "Spain", "MX"),
    ("Cabo Verde", "Saudi Arabia", "US"),
    # Group I
    ("Norway", "France", "US"),
    ("Senegal", "Iraq", "CA"),
    # Group J
    ("Argentina", "Jordan", "US"),
    ("Algeria", "Austria", "US"),
    # Group K
    ("Colombia", "Portugal", "US"),
    ("Congo DR", "Uzbekistan", "US"),
    # Group L
    ("Panama", "England", "US"),
    ("Croatia", "Ghana", "US"),
]

# Get the predictions for all the first round

- We should traing and build one model per country, and that model will be used one time (no reusability)

In [2]:
import pandas as pd

from model_utils import Workflow, predict_match
from tsv_utils import load_country_data


def predict_round(round: list[tuple[str, str, str]], end_date: pd.Timestamp, filename: str | None = None):
    results_df = pd.DataFrame(columns=["team_1", "team_2", "goals_1", "goals_2", "winner", "location"])
    for team_1, team_2, location in round:
        team_1_df = load_country_data(country=team_1, end_date=end_date)
        team_2_df = load_country_data(country=team_2, end_date=end_date)
        goals_1, goals_2 = predict_match(team_1=team_1_df,
                                         team_2=team_2_df,
                                         workflow=Workflow.ATK_DEF,
                                         location=location
                                         )
        winner = team_1 if goals_1 > goals_2 else team_2 if goals_2 > goals_1 else "Draw"
        new_row = pd.DataFrame(
            {"team_1"  : team_1, "team_2": team_2, "goals_1": goals_1, "goals_2": goals_2, "winner": winner,
             "location": location}, index=[0])
        results_df = pd.concat([results_df, new_row], ignore_index=True)

    if filename is not None:
        results_df.to_csv(filename)
    return results_df



In [3]:

end_first_round = pd.to_datetime("2026-06-17")
end_second_round = pd.to_datetime("2026-06-23")
end_third_round = pd.to_datetime("2026-06-27")
first_round_prediction = predict_round(first_round, end_first_round, "first_round_predictions.csv")
second_round_prediction = predict_round(second_round, end_second_round, "second_round_predictions.csv")
third_round_prediction = predict_round(third_round, end_third_round, "third_round_predictions.csv")

In [4]:
def results(round: list[tuple[str, str, str]], start_date, end_date) -> pd.DataFrame:
    results_cols = ["team_1", "team_2", "goals_1", "goals_2", "winner"]
    results_rows = []
    for team_1, team_2, location in round:
        team_1_df = load_country_data(team_1, start_date, end_date)
        team_2_df = load_country_data(team_2, start_date, end_date)
        goals_1 = team_1_df["goals_converted"].iloc[0]
        goals_2 = team_2_df["goals_converted"].iloc[0]
        winner = team_1 if goals_1 > goals_2 else team_2 if goals_2 > goals_1 else "Draw"
        results_rows.append([team_1, team_2, goals_1, goals_2, winner])
    return pd.DataFrame(results_rows, columns=results_cols)


In [5]:
def evaluate_results(results_df: pd.DataFrame, predictions_df: pd.DataFrame) -> tuple[float, float]:
    """
    Returns the accuracy score and the goal error mean of the results compared to the predictions.
    :param results_df: [pd.DataFrame] the results of the match
    :param predictions_df: [pd.DataFrame] the predictions of the model
    :return: [tuple[float, float]] accuracy score and goal error mean
    """
    merged_df = results_df.merge(predictions_df, on=["team_1", "team_2"], how="left")
    merged_df['goal_err'] = (abs(merged_df['goals_1_x'] - merged_df['goals_1_y']) +
                             abs(merged_df['goals_2_x'] - merged_df['goals_2_y'])) / 2
    merged_df['correct_prediction'] = merged_df['winner_x'] == merged_df['winner_y']
    goal_err_mean: float = merged_df['goal_err'].mean()
    accuracy_score: float = merged_df['correct_prediction'].mean()
    return accuracy_score, goal_err_mean


In [6]:
start_date = pd.to_datetime("2026-06-11")
first_round_results = results(first_round, start_date, end_first_round)
first_round_evaluation = evaluate_results(first_round_results, first_round_prediction)
first_round_evaluation

(np.float64(0.5833333333333334), np.float64(0.679628917856672))

In [7]:
second_round_results = results(second_round, end_first_round + pd.Timedelta(days=1), end_second_round)
second_round_evaluation = evaluate_results(second_round_results, second_round_prediction)
second_round_evaluation

(np.float64(0.7083333333333334), np.float64(0.955291678032077))

In [8]:
third_round_results = results(third_round, end_second_round + pd.Timedelta(days=1), end_third_round)
third_round_evaluation = evaluate_results(third_round_results, third_round_prediction)
third_round_evaluation

(np.float64(0.5), np.float64(1.0516318169922638))

In [9]:
# TODO: predicting the round of 32
round_of_32 = [
    # jun 28
    ("South Africa", "Canada", "US"),
    # mon 29
    ("Brazil", "Japan", "US"),
    ("Germany", "Paraguay", "US"),
    ("Netherlands", "Morocco", "MX"),
    # tue 30
    ("Côte d'Ivoire", "Norway", "US"),
    ("France", "Sweden", "US"),
    ("Mexico", "Ecuador", "MX"),
    # wed 01
    ("England", "Congo DR", "US"),
    ("Belgium", "Senegal", "US"),
    ("USA", "Bosnia and Herzegovina", "US"),
    # thu 02
    ("Spain", "Austria", "US"),
    ("Portugal", "Croatia", "CA"),
    ("Switzerland", "Algeria", "CA"),
    # fri 03
    ("Australia", "Egypt", "US"),
    ("Argentina", "Cabo Verde", "US"),
    ("Colombia", "Ghana", "US"),
]
start_round_of_32 = pd.to_datetime("2026-06-28")
round_of_32_pred = predict_round(round_of_32, end_third_round, "round_of_32_predictions.csv")
round_of_32_pred


,team_1,team_2,goals_1,goals_2,winner,location
0,South Africa,Canada,0.287175,2.024047,Canada,US
1,Brazil,Japan,2.239419,1.370329,Brazil,US
2,Germany,Paraguay,3.148853,1.04404,Germany,US
3,Netherlands,Morocco,5.345404,3.359876,Netherlands,MX
4,Côte d'Ivoire,Norway,0.808364,2.75508,Norway,US
5,France,Sweden,2.804579,0.514473,France,US
6,Mexico,Ecuador,0.885567,0.568352,Mexico,MX
7,England,Congo DR,7.576357,2.98162,England,US
8,Belgium,Senegal,0.781941,2.048319,Senegal,US
9,USA,Bosnia and Herzegovina,9.349144,1.388977,USA,US


In [11]:
end_round_of_32 = pd.to_datetime("2026-07-03")
round_of_32_results = results(round=round_of_32, start_date=start_round_of_32, end_date=end_round_of_32)
round_of_32_evaluation = evaluate_results(round_of_32_results, round_of_32_pred)
round_of_32_evaluation

(np.float64(0.625), np.float64(1.5127614716521423))

In [12]:
round_of_16 = [
    # sat 04
    ("Morocco", "Canada", "US"),
    ("France", "Paraguay", "US"),
    # sun 05
    ("Brazil", "Norway", "US"),
    ("Mexico", "England", "MX"),
    # mon 06
    ("Spain", "Portugal", "US"),
    ("USA", "Belgium", "US"),
    # tue 07
    ("Argentina", "Egypt", "US"),
    ("Colombia", "Switzerland", "CA"),
]
round_of_16_pred = predict_round(round_of_16, end_round_of_32, "round_of_16_predictions.csv")
round_of_16_pred


,team_1,team_2,goals_1,goals_2,winner,location
0,Morocco,Canada,3.157125,0.594867,Morocco,US
1,France,Paraguay,2.863061,2.820593,France,US
2,Brazil,Norway,2.254213,2.623809,Norway,US
3,Mexico,England,1.585643,1.583023,Mexico,MX
4,Spain,Portugal,1.790872,0.757475,Spain,US
5,USA,Belgium,1.663799,1.627397,USA,US
6,Argentina,Egypt,1.988217,0.330498,Argentina,US
7,Colombia,Switzerland,0.506435,2.146445,Switzerland,CA


# Discussion
We can start seeing how the countries start having similar goals against each other.
This suggest that the match will be decided by penalty kicks.